# СимургФарм — полное EDA, фармакокинетика, ML и расчёт дозы

**Цель:** по характеристикам пациента предсказать `CL_L_h`, `Vd_L`, затем вычислить конкурсную дозу, удовлетворяющую заданным условиям. Исходные файлы: `expamlomycin_train.csv` (2000 строк) и `expamlomycin_test.csv` (500 строк).

Ноутбук автономен: при запуске воспроизводит EDA, проверку моделей, независимую по обучению валидацию, OOF-калибровку, расчёт дозы и сохранение `submission.csv`. Он не использует неизвестные ответы теста.

> **Ограничение:** экзампломицин — учебный препарат; модель и рассчитанные дозы предназначены **только для конкурсного кейса**, не для медицинского применения. В реальной клинике функцию почек, лекарственные концентрации и дозы оценивают врач и клинический фармаколог.

## 0. Установка и загрузка

В Colab загрузите оба CSV через панель **Файлы**. Если их нет в рабочей папке, следующая ячейка предложит загрузить их напрямую. **Важно:** тестовый CSV имеет разделитель `;`, обучающий — `,`.

In [ ]:
%pip -q install catboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import ks_2samp
from IPython.display import display

sns.set_theme(style='whitegrid')
if not Path('expamlomycin_train.csv').exists() or not Path('expamlomycin_test.csv').exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Положите оба CSV в рабочую директорию ноутбука.')

train = pd.read_csv('expamlomycin_train.csv')
test = pd.read_csv('expamlomycin_test.csv', sep=';')
assert len(train)==2000 and len(test)==500
assert train.patient_id.is_unique and test.patient_id.is_unique
assert 'CL_L_h' in train and 'Vd_L' in train
assert 'CL_L_h' not in test and 'Vd_L' not in test
print('Размеры:', train.shape, test.shape)
display(train.head(3))

## 1. EDA: структура данных, пропуски и различия Train/Test

У обучающей таблицы 14 признаков пациента и 2 таргета. `patient_id` — служебный идентификатор; обучать модель по нему нельзя. Значительная доля пропусков в клинических анализах — повод исследовать доступность признаков, а не удалять пациентов.

In [ ]:
print('Типы данных:')
train.info()
print('Дубликаты строк:',train.duplicated().sum())
print('Категории:')
for c in ['sex','race','oat_variant','sepsis_status']:
    display(train[c].value_counts(dropna=False).rename(c))

miss=pd.DataFrame({'Train, %':100*train.isna().mean(),
                   'Test, %':100*test.isna().mean()}).fillna(0)
miss=miss.loc[miss[['Train, %','Test, %']].max(axis=1)>0].sort_values('Train, %')
display(miss.round(1))
miss.plot.barh(figsize=(9,5));plt.title('Пропуски в двух выборках')
plt.xlabel('% строк');plt.tight_layout();plt.show()

# Доля пропусков зависит от клинического статуса
labs=['crp_mg_l','lactate_mmol_l','albumin_g_l','glucose_mmol_l','ast_u_l']
display((100*train.groupby('sepsis_status')[labs].apply(lambda x:x.isna().mean())).round(1))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
for a,col in zip(ax,['CL_L_h','Vd_L']):
    a.hist(train[col].dropna(), bins=45)
    a.set_title(col+' — распределение')
plt.tight_layout();plt.show()

display(train[['CL_L_h','Vd_L']].describe().round(3))
numcols=train.select_dtypes(include='number').drop(columns=['CL_L_h','Vd_L']).columns
spearman=train[list(numcols)+['CL_L_h','Vd_L']].corr(method='spearman')
fig,ax=plt.subplots(1,2,figsize=(17,7))
sns.heatmap(spearman.loc[numcols,['CL_L_h','Vd_L']],
            annot=True,fmt='.2f',cmap='coolwarm',center=0,ax=ax[0]);ax[0].set_title('Корреляция с таргетами')
sns.heatmap(spearman.loc[numcols,numcols],cmap='coolwarm',center=0,
            vmin=-1,vmax=1,ax=ax[1]);ax[1].set_title('Корреляции между признаками')
plt.tight_layout();plt.show()

## 2. Что говорят биологические показатели

Наблюдаемое в кейсе **не равно** доказанному фармакологическому механизму. Однако есть разумные гипотезы:

- **CL:** оценка функции почек по возрасту, полу, массе и креатинину. Проверим приближение Кокрофта–Голта — это не непосредственный клиренс антибиотика.
- **Vd:** масса тела, сепсис и их взаимодействие; у гидрофильных антибиотиков тяжёлое состояние может менять распределение жидкости.
- **OAT:** в датасете генетический вариант сильно разделяет CL. Это характеристика *условной задачи*; нельзя утверждать, что OAT доказанно определяет клиренс любого реального аминогликозида.
- **Альбумин:** даже после учёта сепсиса возможно небольшое дополнительное объяснение Vd.

Источники: [обзор моделей PK аминогликозидов в ICU](https://pmc.ncbi.nlm.nih.gov/articles/PMC8145041/), [фармакокинетика антибиотиков при критических состояниях](https://pmc.ncbi.nlm.nih.gov/articles/PMC5606477/), [формула Кокрофта–Голта, NKF](https://www.kidney.org/professionals/kdoqi/cockcroft-gault-equation-estimating-creatinine-clearance). В критическом состоянии при нестационарном креатинине оценочные формулы могут давать ошибку.

In [ ]:
train_eda=train.copy()
train_eda['CrCl_est']=(140-train.age)*train.weight_kg/(.814*train.scr_umol_l)*np.where(train.sex=='F',.85,1.)
train_eda['BSA']=np.sqrt(train.height_cm*train.weight_kg/3600)
train_eda['weight_x_sepsis']=train.weight_kg*train.sepsis_status
print('Spearman для инженерных признаков:')
display(train_eda[['CrCl_est','BSA','weight_x_sepsis','CL_L_h','Vd_L']]
        .corr(method='spearman').loc[['CrCl_est','BSA','weight_x_sepsis'],['CL_L_h','Vd_L']].round(3))

fig,ax=plt.subplots(2,2,figsize=(12,9))
sns.scatterplot(data=train_eda,x='CrCl_est',y='CL_L_h',alpha=.25,ax=ax[0,0])
sns.boxplot(data=train_eda,x='oat_variant',y='CL_L_h',ax=ax[0,1])
sns.boxplot(data=train_eda,x='sepsis_status',y='Vd_L',ax=ax[1,0])
sns.scatterplot(data=train_eda,x='weight_kg',y='Vd_L',hue='sepsis_status',alpha=.35,ax=ax[1,1])
plt.tight_layout();plt.show()

import statsmodels.formula.api as smf
ols_vd=smf.ols('Vd_L ~ weight_kg * sepsis_status', data=train_eda).fit()
print('Простая объяснимая модель Vd, R² (in-sample):',round(ols_vd.rsquared,3))
print(ols_vd.params.round(4))

## 3. Сдвиг между Train и Test

Сравниваем распределения признаков. Проверка Колмогорова–Смирнова не доказывает идентичности распределений и при многочисленных тестах может давать ложноположительные результаты. В исходных файлах особенно стоит следить за лактатом.

In [ ]:
shift=[]
for col in test.select_dtypes(include='number').columns:
    a=train[col].dropna();b=test[col].dropna()
    ks=ks_2samp(a,b)
    shift.append([col,round(float(ks.statistic),4),round(float(ks.pvalue),5),len(a),len(b)])
shift=pd.DataFrame(shift,columns=['feature','KS statistic','p-value','N train','N test'])
display(shift.sort_values('p-value'))

## 4. Анализ доступности признаков

Здесь результаты последовательного абляционного анализа на 1600 пациентах (5-fold CV; MAPE, %), полученные в исследовании. Минимальная модель для Vd: вес, сепсис и их взаимодействие. Альбумин даёт небольшой прирост, тогда как исключение OAT или функции почек для CL приводит к заметной потере точности.

| Группа признаков | MAPE, % |
|:--|--:|
| CL: CrCl + OAT | 13,91 |
| CL: CrCl без OAT | 20,81 |
| CL: без креатинина, с OAT + базовые характеристики | 20,03 |
| Vd: только масса | 13,09 |
| Vd: масса + сепсис + взаимодействие | 9,77 |
| Vd: + альбумин | 9,58 |

Эти числа относятся к конкретным моделям и разбиению. Они **не** являются доказательством клинической пригодности или экономической эквивалентности анализов. Сокращение признаков — компромисс точности, времени и доступности измерений.

## 5. Алгоритм: несколько моделей для разных таргетов

**CL:** прогнозируем `log(CL)` по `log(CrCl_est)` и OAT-варианту. Если креатинин неизвестен, сначала восстанавливаем его отдельным CatBoost по доступным признакам, **не используя истинные CL/Vd**. Небольшой вклад второго CatBoost — ансамбль по CV.

**Vd:** взвешенная медианная линейная регрессия по весу, сепсису, их взаимодействию и альбумину. Вес каждого пациента `1/Vd` приближает обучение к конкурсной MAPE. Дополняем небольшим CatBoost.

В качестве дополнительных альтернатив на этапе исследования проверены обычная Ridge/LinearRegression, ExtraTrees, HistGradientBoosting, LightGBM и XGBoost. Для данного набора простая физиологическая модель оказалась более устойчива, чем большинство бустингов; глубокая модель здесь не обязательна.

Ниже — **полный исполняемый код**, все предобработки обучаются заново внутри каждого фолда.

In [ ]:
"""SimurgPharm / examplomycin case. Training and submission pipeline.
Competition-only educational model. NOT a dosing tool for patients.
Python 3.10+, pandas/scipy/sklearn/catboost.
"""
from __future__ import annotations
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from scipy.optimize import minimize_scalar
from sklearn.impute import SimpleImputer
from sklearn.linear_model import QuantileRegressor, Ridge
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error

RANDOM_STATE = 42
CATS = ["sex", "oat_variant"]
DROP_RAW = ["patient_id", "race", "CL_L_h", "Vd_L"]
SCR_FEATURES = ["age", "sex", "height_cm", "weight_kg", "lean_bw_kg",
                "albumin_g_l", "crp_mg_l", "sepsis_status", "oat_variant",
                "glucose_mmol_l", "lactate_mmol_l", "ast_u_l"]


def cat_input(df: pd.DataFrame) -> pd.DataFrame:
    x = df.drop(columns=DROP_RAW, errors="ignore").copy()
    for col in CATS:
        x[col] = x[col].fillna("Missing").astype(str)
    return x


def imputer_input(df: pd.DataFrame) -> pd.DataFrame:
    x = df[SCR_FEATURES].copy()
    for col in CATS:
        x[col] = x[col].fillna("Missing").astype(str)
    return x


def cl_features(df: pd.DataFrame, creatinine: np.ndarray) -> pd.DataFrame:
    scr = np.asarray(creatinine, dtype=float)
    cg = ((140 - df["age"].to_numpy()) * df["weight_kg"].to_numpy()
          / (0.814 * np.maximum(scr, 10))
          * np.where(df["sex"].to_numpy() == "F", 0.85, 1.0))
    x = pd.DataFrame({
        "log_CrCl_CG": np.log(np.maximum(cg, 1.e-3)),
        "OAT_Rapid": (df["oat_variant"].to_numpy() == "Rapid").astype(int),
        "OAT_Slow": (df["oat_variant"].to_numpy() == "Slow").astype(int),
    }, index=df.index)
    return x


def vd_features(df: pd.DataFrame) -> pd.DataFrame:
    wt = df["weight_kg"].to_numpy()
    sep = df["sepsis_status"].to_numpy()
    return pd.DataFrame({
        "weight_kg": wt,
        "sepsis_status": sep,
        "weight_x_sepsis": wt * sep,
        "albumin_g_l": df["albumin_g_l"].to_numpy(),
    }, index=df.index)


def train_models(df_train: pd.DataFrame):
    """Fit SCR imputation + physiological models and boosted residual complements.
    IMPORTANT: estimation of missing creatinine uses only observed PATIENT features,
    not CL or Vd, and fits on train partition of each fold.
    """
    scr_features = imputer_input(df_train)
    scr_obs = df_train["scr_umol_l"].notna().to_numpy()
    scr_model = CatBoostRegressor(
        iterations=600, depth=4, learning_rate=0.04, loss_function="RMSE",
        random_seed=RANDOM_STATE, thread_count=4, verbose=False
    )
    scr_model.fit(scr_features.loc[scr_obs], df_train.loc[scr_obs,"scr_umol_l"], cat_features=CATS)

    scr = df_train["scr_umol_l"].to_numpy(copy=True)
    if (~scr_obs).any():
        scr[~scr_obs] = np.maximum(scr_model.predict(scr_features.loc[~scr_obs]), 10.)

    cl_model = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.)
    )
    cl_model.fit(cl_features(df_train, scr), np.log(df_train["CL_L_h"]))

    vd_model = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        QuantileRegressor(quantile=.5, alpha=.005, solver="highs")
    )
    # For linear predictions: MAE weighted by 1/y is exactly the MAPE objective.
    vd_model.fit(vd_features(df_train), df_train["Vd_L"],
                 quantileregressor__sample_weight=1 / df_train["Vd_L"].to_numpy())

    Xcat = cat_input(df_train)
    kwargs=dict(iterations=950, depth=4, learning_rate=.035, l2_leaf_reg=4,
                loss_function="RMSE", random_seed=RANDOM_STATE,
                thread_count=4, verbose=False)
    cat_cl=CatBoostRegressor(**kwargs)
    cat_vd=CatBoostRegressor(**kwargs)
    cat_cl.fit(Xcat, np.log(df_train["CL_L_h"]), cat_features=CATS)
    cat_vd.fit(Xcat, np.log(df_train["Vd_L"]), cat_features=CATS)
    return {"scr_model":scr_model,"cl_model":cl_model,"vd_model":vd_model,
            "cat_cl":cat_cl,"cat_vd":cat_vd}


def component_predictions(models: dict, df_pred: pd.DataFrame) -> dict:
    scr = df_pred["scr_umol_l"].to_numpy(copy=True)
    missing = np.isnan(scr)
    if missing.any():
        scr[missing] = np.maximum(
            models["scr_model"].predict(imputer_input(df_pred).loc[missing]), 10.)
    cp = np.exp(models["cl_model"].predict(cl_features(df_pred, scr)))
    vp = models["vd_model"].predict(vd_features(df_pred))
    catx=cat_input(df_pred)
    cp_cat=np.exp(models["cat_cl"].predict(catx))
    vp_cat=np.exp(models["cat_vd"].predict(catx))
    return {"phys_CL":cp,"phys_Vd":np.maximum(vp,0.1),"cat_CL":cp_cat,
            "cat_Vd":vp_cat,"scr_missing":missing}


def combine_predictions(pred: dict, cl_weight=0.10, vd_weight=0.30):
    """CL tree weight is zero when creatinine is missing: learned via CV."""
    wcl=np.where(pred["scr_missing"],0.,cl_weight)
    cl=np.exp((1-wcl)*np.log(pred["phys_CL"])+wcl*np.log(pred["cat_CL"]))
    vd=np.exp((1-vd_weight)*np.log(pred["phys_Vd"])
              +vd_weight*np.log(pred["cat_Vd"]))
    return cl,vd


def cv_predictions(df_train: pd.DataFrame, n_splits=5, verbose=True):
    cv=KFold(n_splits=n_splits,shuffle=True,random_state=RANDOM_STATE)
    cl_oof=np.full(len(df_train),np.nan);vd_oof=np.full(len(df_train),np.nan)
    for fold,(tr_idx,vl_idx) in enumerate(cv.split(df_train),start=1):
        mt=train_models(df_train.iloc[tr_idx])
        p=component_predictions(mt,df_train.iloc[vl_idx])
        cl_oof[vl_idx],vd_oof[vl_idx] = combine_predictions(p)
        if verbose:print(f"  Fold {fold}/{n_splits} completed",flush=True)
    return cl_oof,vd_oof


def mape_percent(actual, pred):
    return 100*mean_absolute_percentage_error(actual,pred)


def calibrate_scale(y, pred):
    """Minimize train OOF MAPE with one multiplicative coefficient."""
    result=minimize_scalar(lambda a:np.mean(np.abs(y-a*pred)/y),
                           bounds=(.85,1.05),method="bounded")
    return float(result.x)


def dose_bounds(cl, vd):
    """Concentration model C(t)=Dose/Vd*exp(-CL*t/Vd). Units mg, L, h."""
    cl=np.maximum(np.asarray(cl,dtype=float),.001)
    vd=np.maximum(np.asarray(vd,dtype=float),.001)
    q = -np.expm1(-12*cl/vd)
    lower=72*cl/q # average(0-12) >= 6
    upper=np.minimum(144*cl/q,26*vd*np.exp(.5*cl/vd)) # avg <= 12 AND peak<=26
    return lower,upper


def success_rate(cl_true, vd_true, dose):
    cl=np.asarray(cl_true,dtype=float);vd=np.asarray(vd_true,dtype=float)
    dose=np.asarray(dose,dtype=float)
    c05=(dose/vd)*np.exp(-.5*cl/vd)
    cavg=(dose/cl)*(-np.expm1(-12*cl/vd))/12
    meets=(c05 <= 26 + 1e-9)&(cavg >= 6 - 1e-9)&(cavg <= 12 + 1e-9)
    return 100*meets.mean(),meets


def robust_dose(cl, vd, errors, grid_n=151):
    """Maximize empirical success probability, honoring nominal constraints.

    errors: two-column array of OOF log(true/pred) errors.
    In rare infeasible cases for point predictions, return peak-bound dose and
    separately flag infeasibility: no dose can satisfy both nominal conditions.
    """
    fac_cl=np.exp(errors[:,0]);fac_vd=np.exp(errors[:,1])
    lo,hi=dose_bounds(cl,vd)
    dose=np.zeros(len(lo));prob=np.zeros(len(lo));ok=lo<=hi
    for i in range(len(lo)):
        if not ok[i]:
            dose[i]=hi[i]
            continue
        simlo,simhi=dose_bounds(cl[i]*fac_cl,vd[i]*fac_vd)
        candidates=np.linspace(lo[i],hi[i],grid_n)
        hit=((simlo[None,:] <= candidates[:,None]) &
             (candidates[:,None] <= simhi[None,:])).mean(axis=1)
        best=np.flatnonzero(hit >= hit.max() - 1.e-12)
        dose[i]=np.median(candidates[best]);prob[i]=hit.max()
    return dose,prob,ok


def score_summary(cl_true, vd_true, cl_pred, vd_pred, dose):
    mcl=mape_percent(cl_true,cl_pred);mvd=mape_percent(vd_true,vd_pred)
    tar=success_rate(cl_true,vd_true,dose)[0]
    p_kacc=max(0.,100-.5*(mcl+mvd))
    return {"CL_MAPE_pct":round(mcl,4),"Vd_MAPE_pct":round(mvd,4),
            "Mean_MAPE_pct":round((mcl+mvd)/2,4),
            "TAR_pct":round(tar,4),"P_kacc":round(p_kacc,4),
            "Technical_T":round((p_kacc+tar)/2,4)}


def model_report(df_train, test, output_dir, do_holdout=True):
    """All CV/holdout predictions are strictly out-of-fold; test labels unused."""
    from pathlib import Path
    import json,joblib
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    ycl=df_train["CL_L_h"].to_numpy();yvd=df_train["Vd_L"].to_numpy()
    result={"schema":"patient_id,Vd_L,CL_L_h,Dose_mg", "note":"Contest-only. Not medical dosing advice."}

    if do_holdout:
        tidx,vidx=train_test_split(np.arange(len(df_train)),test_size=.20,
                                   random_state=RANDOM_STATE)
        subset=df_train.iloc[tidx].reset_index(drop=True)
        print("CV on 1600 training rows, holdout 400 untouched:",flush=True)
        c_oof,v_oof=cv_predictions(subset)
        sc=calibrate_scale(subset["CL_L_h"].to_numpy(),c_oof)
        sv=calibrate_scale(subset["Vd_L"].to_numpy(),v_oof)
        eps=np.column_stack([np.log(subset["CL_L_h"].to_numpy()/(sc*c_oof)),
                             np.log(subset["Vd_L"].to_numpy()/(sv*v_oof))])
        models=train_models(df_train.iloc[tidx])
        pred=component_predictions(models,df_train.iloc[vidx]);c,v=combine_predictions(pred)
        c*=sc;v*=sv
        doses,probs,feasible=robust_dose(c,v,eps)
        result['holdout']=score_summary(ycl[vidx],yvd[vidx],c,v,doses)
        result['holdout']['optimal_feasible_pct']=round(100*np.mean(dose_bounds(ycl[vidx],yvd[vidx])[0]<=dose_bounds(ycl[vidx],yvd[vidx])[1]),3)
        result['holdout']['predicted_feasible_pct']=round(100*feasible.mean(),3)
        print('Holdout:',result['holdout'],flush=True)
        pd.DataFrame({"patient_id":df_train.iloc[vidx].patient_id.to_numpy(),
                      "CL_true":ycl[vidx],"Vd_true":yvd[vidx],
                      "CL_pred":c,"Vd_pred":v,"Dose_mg":doses}).to_csv(out/'holdout_predictions.csv',index=False)

    print("Full 5fold OOF for final calibration:",flush=True)
    pc,pv=cv_predictions(df_train)
    sc=calibrate_scale(ycl,pc);sv=calibrate_scale(yvd,pv)
    pc*=sc;pv*=sv
    residual=np.column_stack([np.log(ycl/pc),np.log(yvd/pv)])
    dp,_,ok=robust_dose(pc,pv,residual)
    result['full_cv']=score_summary(ycl,yvd,pc,pv,dp)
    result['full_cv']['predicted_feasible_pct']=round(100*ok.mean(),3)
    result['scale_CL']=sc;result['scale_Vd']=sv
    print('Full OOF:',result['full_cv'], 'scales',sc,sv,flush=True)

    print('Fitting all 2000 and predicting 500...',flush=True)
    models=train_models(df_train)
    p=component_predictions(models,test);cl,vd=combine_predictions(p)
    cl*=sc;vd*=sv
    dose,conf,feasible=robust_dose(cl,vd,residual)
    assert (np.isfinite(cl)&np.isfinite(vd)&np.isfinite(dose)).all()
    assert (cl>0).all() and (vd>0).all() and (dose>0).all()
    assert len(test)==len(np.unique(test.patient_id))
    submission=pd.DataFrame({"patient_id":test.patient_id.to_numpy(),
                             "Vd_L":vd,"CL_L_h":cl,"Dose_mg":dose})
    submission.to_csv(out/'submission.csv',index=False,float_format='%.6f')
    result['submission_summary']={
        'rows':len(submission),'duplicate_ids':int(submission.patient_id.duplicated().sum()),
        'nan_predictions':int(submission[['Vd_L','CL_L_h','Dose_mg']].isna().sum().sum()),
        'predicted_dose_feasibility_pct':round(float(feasible.mean()*100),3),
        'predicted_infeasible_count':int((~feasible).sum()),
        'median_dose_mg':round(float(submission.Dose_mg.median()),3),
        'min_dose_mg':round(float(submission.Dose_mg.min()),3),
        'max_dose_mg':round(float(submission.Dose_mg.max()),3)}
    result['features_for_CL']=['age','sex','weight_kg','scr_umol_l','oat_variant']
    result['features_for_Vd']=['weight_kg','sepsis_status','albumin_g_l']
    result['clinical_labs_dosing_warning']='Fictional drug, model not clinically validated.'
    with open(out/'validation_report.json','w',encoding='utf8') as f:
        json.dump(result,f,ensure_ascii=False,indent=2)
    joblib.dump({'models':models,'scale_CL':sc,'scale_Vd':sv,
                 'residuals':residual,'cv_report':result['full_cv']},out/'fitted_models.joblib')
    print('Submission summary:',result['submission_summary'],flush=True)
    return result,submission




## 6. Валидация и расчёт дозы

Мы выполняем две проверки:

1. **400 пациентов отдельно от обучения моделей**; на остальных 1600 строим пять OOF-предсказаний для калибровки. Из-за ранее проводившегося EDA это не абсолютно нетронутый внешний тест.
2. **5-fold OOF на всех 2000** — для калибровки финального ансамбля и оценки типичных ошибок. Это также используется для выбора дозы.

Формула из задания:

\[ C(t)=\frac{Dose}{Vd}\exp\left(-\frac{CL}{Vd}t\right).\]

Из ограничения средней концентрации и потолка `C(0.5)` получаем интервал:

\[D_{min}=\frac{72CL}{1-e^{-12CL/Vd}},\quad D_{max}=\min\left(\frac{144CL}{1-e^{-12CL/Vd}},26Vd\,e^{0.5CL/Vd}\right).\]

Если он существует, `robust_dose()` выбирает дозу **внутри** него, максимизирующую эмпирическую вероятность выполнения условий на основе OOF-ошибок. При пустом интервале модель записывает верхнюю границу концентрации, но не скрывает несовместимость двух условий.

In [ ]:
from pathlib import Path
report, submission = model_report(train, test, Path('.'), do_holdout=True)
print('Итоговые показатели на 400 пациентах:')
display(pd.DataFrame([report['holdout']],index=['Holdout']).T)
print('OOF на 2000 пациентах:')
display(pd.DataFrame([report['full_cv']],index=['5-fold OOF']).T)
print('Предсказания для теста:')
display(submission.head())

## 7. Разбор ошибок и проверка CSV перед отправкой

Проверяем ошибку в клинически важных группах, а не только среднюю метрику. Особенно важно помнить, что при отсутствии креатинина неопределённость CL выше. Обозначение `Dose_mg` выбрано по смыслу PDF; если площадка требует другой точный заголовок, переименуйте столбец согласно её шаблону.

In [ ]:
holdout = pd.read_csv('holdout_predictions.csv')
for col in ['CL','Vd']:
    holdout[col+'_APE_%'] = 100 * (holdout[col+'_true'] - holdout[col+'_pred']).abs() / holdout[col+'_true']
clinical = holdout.merge(train[['patient_id','oat_variant','sepsis_status','scr_umol_l']],on='patient_id')
print('Ошибка в зависимости от наличия креатинина:')
display(clinical.groupby(clinical.scr_umol_l.isna())[['CL_APE_%','Vd_APE_%']].agg(['mean','count']).round(2))
print('Ошибка по OAT:')
display(clinical.groupby('oat_variant')[['CL_APE_%','Vd_APE_%']].mean().round(2))

fig, ax=plt.subplots(1,3,figsize=(15,4))
for i,col in enumerate(['CL','Vd']):
    ax[i].scatter(holdout[col+'_true'],holdout[col+'_pred'],alpha=.3,s=12)
    lower=min(holdout[col+'_true'].min(),holdout[col+'_pred'].min())
    upper=max(holdout[col+'_true'].max(),holdout[col+'_pred'].max())
    ax[i].plot([lower,upper],[lower,upper],'r--',lw=1)
    ax[i].set(xlabel='Истина',ylabel='Прогноз',title=col)
ax[2].hist(submission['Dose_mg'], bins=35)
ax[2].set_title('Распределение конкурсных доз, мг')
plt.tight_layout();plt.show()

assert submission.columns.tolist() == ['patient_id','Vd_L','CL_L_h','Dose_mg']
assert submission.patient_id.is_unique
assert submission[['Vd_L','CL_L_h','Dose_mg']].notna().all().all()
assert (submission[['Vd_L','CL_L_h','Dose_mg']]>0).all().all()
assert submission.patient_id.tolist() == test.patient_id.tolist()
print('CSV проверен; строки:',len(submission))

## 8. Сохранение результатов

Файл **`submission.csv`** содержит только `patient_id,Vd_L,CL_L_h,Dose_mg` в порядке оригинального теста. Также сохранены `validation_report.json`, `fitted_models.joblib`, `holdout_predictions.csv`. Для повторного анализа можно запустить ноутбук с теми же CSV.

Важная методологическая оговорка: доля корректных доз и MAPE на скрытом тесте **неизвестны**. Результаты кросс-валидации не гарантируют точность на реальных пациентах. Полностью индивидуализированное назначение аминогликозидов в реальном мире не должно основываться на этом кейсе.

In [ ]:
print('Результат сохранён:',Path('submission.csv').resolve())
try:
    from google.colab import files
    files.download('submission.csv')
except ImportError:
    pass

### Первоисточники и рамки интерпретации

- Конкурсное задание «СимургФарм» (загруженный PDF): требования к `CL`, `Vd`, ограничения дозы, MAPE/TAR и формат CSV.
- [Aminoglycosides in the Intensive Care Unit: What Is New in Population PK Modeling?](https://pmc.ncbi.nlm.nih.gov/articles/PMC8145041/) — о значимости массы и функции почек.
- [Pharmacokinetic considerations and dosing strategies of antibiotics in the critically ill patient](https://pmc.ncbi.nlm.nih.gov/articles/PMC5606477/) — о влиянии сепсиса на фармакокинетику.
- [National Kidney Foundation: Cockcroft–Gault Equation](https://www.kidney.org/professionals/kdoqi/cockcroft-gault-equation-estimating-creatinine-clearance) — о расчётном клиренсе креатинина.
- [Drug transport by Organic Anion Transporters](https://pubmed.ncbi.nlm.nih.gov/22841915/) — контекст механизмов OAT, **не доказательство** механизма условного экзампломицина.